# Chapter 13: Putting it all in practice
## A real-life example of data engineering and machine learning

### Importing packages

In [1]:
import pandas
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

## 13.1 The Titanic dataset

### Loading the dataset

For the next two cells, please run only one of them.
- Run the first cell if you cloned the Github Repo
- Run the second cell if you opened this as a Google Colab

In [2]:
# IMPORTANT: ONLY RUN THIS CELL IF YOU HAVE CLONED THE REPO
raw_data = pandas.read_csv('./titanic.csv')
raw_data

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [ ]:
# IMPORTANT: ONLY RUN THIS CELL IF YOU ARE WORKING ON A COLAB
url = "https://raw.githubusercontent.com/luisguiserrano/manning/master/Chapter_13_End_to_end_example/titanic.csv"
raw_data = pandas.read_csv(url)
raw_data

### Studying the dataset

In [4]:
len(raw_data)

891

In [5]:
raw_data['Survived']

0      0
1      1
2      1
3      1
4      0
      ..
886    0
887    1
888    0
889    1
890    0
Name: Survived, Length: 891, dtype: int64

In [6]:
sum(raw_data['Survived'])

342

## 13.2 Cleaning up our dataset: Missing values and how to deal with them

In [7]:
raw_data['Cabin']

0       NaN
1       C85
2       NaN
3      C123
4       NaN
       ... 
886     NaN
887     B42
888     NaN
889    C148
890     NaN
Name: Cabin, Length: 891, dtype: str

In [8]:
raw_data.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

### Dropping columns with missing data

The Cabin column is missing too many values to be useful (687 out of 891). Let's drop it.

In [9]:
clean_data = raw_data.drop('Cabin', axis=1)

### Filling in missing data

- For the Age column, we fill in the missing values with the median of all ages.
- For the Embarked column, we make a new category called 'U', for Unknown port of embarkment.

In [10]:
clean_data['Age']

0      22.0
1      38.0
2      26.0
3      35.0
4      35.0
       ... 
886    27.0
887    19.0
888     NaN
889    26.0
890    32.0
Name: Age, Length: 891, dtype: float64

In [11]:
median_age = clean_data['Age'].median()
print('Median age:', median_age)  # 28.0
clean_data['Age'] = clean_data['Age'].fillna(median_age)
clean_data['Embarked'] = clean_data['Embarked'].fillna('U')
clean_data.isna().sum()

Median age: 28.0


PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

Saving the clean dataset (the book skips this step).

In [12]:
clean_data.to_csv('./clean_titanic_data.csv', index=None)

## 13.3 Feature engineering: Transforming the features in our dataset before training the models

In [13]:
preprocessed_data = pandas.read_csv('./clean_titanic_data.csv')
preprocessed_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


### One-hot encoding

`get_dummies` with `columns=` creates the new columns, attaches them to the dataset, and removes the original ones. We use `dtype=int` so that the new columns are 0 and 1 (without it, recent versions of pandas give True and False).

In [14]:
preprocessed_data = pandas.get_dummies(preprocessed_data,
    columns=['Sex', 'Embarked'], dtype=int)
preprocessed_data.head()

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_U
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,0,1,0,0,1,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,1,0,1,0,0,0
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,1,0,0,0,1,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,1,0,0,0,1,0
4,5,0,3,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,0,1,0,0,1,0


### Can we one-hot encode numerical features? The Pclass column

In [15]:
for pclass in [1, 2, 3]:
    passengers = preprocessed_data[preprocessed_data['Pclass'] == pclass]
    survival_rate = 100 * passengers['Survived'].mean()
    print(f'In class {pclass}, {survival_rate:.2f}% of the passengers survived')
# Expected: 62.96% (first), 47.28% (second), 24.24% (third)

In class 1, 62.96% of the passengers survived
In class 2, 47.28% of the passengers survived
In class 3, 24.24% of the passengers survived


In [16]:
preprocessed_data = pandas.get_dummies(preprocessed_data,
    columns=['Pclass'], dtype=int)
preprocessed_data.head()

,PassengerId,Survived,Name,Age,SibSp,Parch,Ticket,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_U,Pclass_1,Pclass_2,Pclass_3
0,1,0,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,0,1,0,0,1,0,0,0,1
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,1,0,1,0,0,0,1,0,0
2,3,1,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,1,0,0,0,1,0,0,0,1
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,1,0,0,0,1,0,1,0,0
4,5,0,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,0,1,0,0,1,0,0,0,1


### Binning the Age column

`pandas.cut` splits the ages into the intervals (0, 10], (10, 20], ..., (70, 80]. Then we one-hot encode these eight intervals.

In [17]:
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80]
preprocessed_data['Age'] = pandas.cut(preprocessed_data['Age'], bins)
preprocessed_data['Age'].value_counts(sort=False)

Age
(0, 10]      64
(10, 20]    115
(20, 30]    407
(30, 40]    155
(40, 50]     86
(50, 60]     42
(60, 70]     17
(70, 80]      5
Name: count, dtype: int64

In [18]:
preprocessed_data = pandas.get_dummies(preprocessed_data,
    columns=['Age'], dtype=int)
preprocessed_data.head()

,PassengerId,Survived,Name,SibSp,Parch,Ticket,Fare,Sex_female,Sex_male,Embarked_C,...,Pclass_2,Pclass_3,"Age_(0, 10]","Age_(10, 20]","Age_(20, 30]","Age_(30, 40]","Age_(40, 50]","Age_(50, 60]","Age_(60, 70]","Age_(70, 80]"
0,1,0,"Braund, Mr. Owen Harris",1,0,A/5 21171,7.2500,0,1,0,...,0,1,0,0,1,0,0,0,0,0
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,0,PC 17599,71.2833,1,0,1,...,0,0,0,0,0,1,0,0,0,0
2,3,1,"Heikkinen, Miss. Laina",0,0,STON/O2. 3101282,7.9250,1,0,0,...,0,1,0,0,1,0,0,0,0,0
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,0,113803,53.1000,1,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,0,"Allen, Mr. William Henry",0,0,373450,8.0500,0,1,0,...,0,1,0,0,0,1,0,0,0,0


### Feature selection

In [19]:
preprocessed_data = preprocessed_data.drop(['Name', 'Ticket', 'PassengerId'],
    axis=1)
print(preprocessed_data.shape)  # (891, 21)
preprocessed_data.head()

(891, 21)


,Survived,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_U,...,Pclass_2,Pclass_3,"Age_(0, 10]","Age_(10, 20]","Age_(20, 30]","Age_(30, 40]","Age_(40, 50]","Age_(50, 60]","Age_(60, 70]","Age_(70, 80]"
0,0,1,0,7.2500,0,1,0,0,1,0,...,0,1,0,0,1,0,0,0,0,0
1,1,1,0,71.2833,1,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,1,0,0,7.9250,1,0,0,0,1,0,...,0,1,0,0,1,0,0,0,0,0
3,1,1,0,53.1000,1,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
4,0,0,0,8.0500,0,1,0,0,1,0,...,0,1,0,0,0,1,0,0,0,0


Saving the preprocessed dataset (the book skips this step).

In [20]:
preprocessed_data.to_csv('./preprocessed_titanic_data.csv', index=None)

## 13.4 Training our models

### Splitting the data into features and labels, and training, validation, and testing

In [21]:
preprocessed_data = pandas.read_csv('./preprocessed_titanic_data.csv')
features = preprocessed_data.drop(['Survived'], axis=1)
labels = preprocessed_data['Survived']

In [22]:
# We fix random_state to make sure we always get the same split
features_train, features_validation_test, labels_train, labels_validation_test = train_test_split(
    features, labels, test_size=0.4, random_state=100)
features_validation, features_test, labels_validation, labels_test = train_test_split(
    features_validation_test, labels_validation_test, test_size=0.5, random_state=100)
print(len(features_train), len(features_validation), len(features_test))  # 534 178 179

534 178 179


### Training several models on our dataset

In the notebook, we fix `random_state=0` in the models that use randomness, so that you always get the same results as the book.

In [23]:
# The book uses LogisticRegression(). We add max_iter=1000 (instead of the default 100)
# so the training converges and scikit-learn doesn't show a warning. The results are the same.
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(features_train, labels_train)
dt_model = DecisionTreeClassifier(random_state=0)
dt_model.fit(features_train, labels_train)
nb_model = GaussianNB()
nb_model.fit(features_train, labels_train)
svm_model = SVC()
svm_model.fit(features_train, labels_train)
rf_model = RandomForestClassifier(random_state=0)
rf_model.fit(features_train, labels_train)
gb_model = GradientBoostingClassifier(random_state=0)
gb_model.fit(features_train, labels_train)
ab_model = AdaBoostClassifier(random_state=0)
ab_model.fit(features_train, labels_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given at each `estimator` at eachboosting iteration.Thus, it is only used when `estimator` exposes a `random_state`.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",0
,"estimator estimator: object, default=NoneThe base estimator from which the boosted ensemble is built.Support for sample weighting is required, as well as proper``classes_`` and ``n_classes_`` attributes. If ``None``, thenthe base estimator is :class:`~sklearn.tree.DecisionTreeClassifier`initialized with `max_depth=1`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",None
,"n_estimators n_estimators: int, default=50The maximum number of estimators at which boosting is terminated.In case of perfect fit, the learning procedure is stopped early.Values must be in the range `[1, inf)`.",50
,"learning_rate learning_rate: float, default=1.0Weight applied to each classifier at each boosting iteration. A higherlearning rate increases the contribution of each classifier. There isa trade-off between the `learning_rate` and `n_estimators` parameters.Values must be in the range `(0.0, inf)`.",1.0
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels.","ndarray[int64](2,)","[0,1]"
estimator_ estimator_: estimatorThe base estimator from which the ensemble is grown... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,DecisionTreeClassifier,DecisionTreeC...r(max_depth=1)
estimator_errors_ estimator_errors_: ndarray of floatsClassification error for each estimator in the boostedensemble.,"ndarray[float64](50,)","[0.19,0.33,0.41,...,0.49,0.49,0.49]"
estimator_weights_ estimator_weights_: ndarray of floatsWeights for each estimator in the boosted ensemble.,"ndarray[float64](50,)","[1.42,0.73,0.38,...,0.04,0.04,0.04]"
estimators_ estimators_: list of classifiersThe collection of fitted sub-estimators.,list,"[DecisionTreeC...ate=209652396), DecisionTreeC...ate=398764591), DecisionTreeC...ate=924231285), DecisionTreeC...te=1478610112), ...]"
"feature_importances_ feature_importances_: ndarray of shape (n_features,)The impurity-based feature importances if supported by the``estimator`` (when based on decision trees).Warning: impurity-based feature importances can be misleading forhigh cardinality features (many unique values). See:func:`sklearn.inspection.permutation_importance` as an alternative.","ndarray[float64](20,)","[0.12,0.02,0.4 ,...,0. ,0. ,0. ]"


In the notebook, we put the seven models in a dictionary, so we can evaluate them all with a loop.

In [24]:
models = {'Logistic regression': lr_model, 'Decision tree': dt_model, 'Naive Bayes': nb_model,
          'SVM': svm_model, 'Random forest': rf_model, 'Gradient boosting': gb_model, 'AdaBoost': ab_model}

### Evaluating the models

#### Accuracy

In [25]:
lr_model.score(features_validation, labels_validation)

0.7696629213483146

In [26]:
print('Accuracy of the models on the validation set:')
for name, model in models.items():
    print(f'{name}: {model.score(features_validation, labels_validation):.2f}')

Accuracy of the models on the validation set:
Logistic regression: 0.77
Decision tree: 0.78
Naive Bayes: 0.75
SVM: 0.68
Random forest: 0.78
Gradient boosting: 0.81
AdaBoost: 0.74


#### F1-score

In [27]:
lr_predicted_labels = lr_model.predict(features_validation)
f1_score(labels_validation, lr_predicted_labels)

0.6870229007633588

In [28]:
print('F1-scores of the models on the validation set:')
for name, model in models.items():
    predicted_labels = model.predict(features_validation)
    print(f'{name}: {f1_score(labels_validation, predicted_labels):.2f}')

F1-scores of the models on the validation set:
Logistic regression: 0.69
Decision tree: 0.71
Naive Bayes: 0.68
SVM: 0.40
Random forest: 0.71
Gradient boosting: 0.74
AdaBoost: 0.65


### Testing the model

Accuracy and F1-score of the gradient boosting model on the testing set.

In [29]:
gb_model.score(features_test, labels_test)

0.8324022346368715

In [30]:
gb_predicted_test_labels = gb_model.predict(features_test)
f1_score(labels_test, gb_predicted_test_labels)

0.8026315789473685

## 13.5 Tuning the hyperparameters to find the best model: Grid search

### Grid search by hand (table 13.4)

In [31]:
for C in [1, 10]:
    for gamma in [0.1, 1, 10]:
        svm = SVC(kernel='rbf', C=C, gamma=gamma)
        svm.fit(features_train, labels_train)
        print(C, gamma, svm.score(features_validation, labels_validation))

1 0.1 0.702247191011236
1 1 0.6966292134831461
1 10 0.6685393258426966
10 0.1 0.7247191011235955
10 1 0.6910112359550562
10 10 0.651685393258427


### Grid search with GridSearchCV

In [32]:
svm_parameters = {'kernel': ['rbf'],
                  'C': [0.01, 0.1, 1, 10, 100],
                  'gamma': [0.01, 0.1, 1, 10, 100]}
svm = SVC()
svm_gs = GridSearchCV(estimator=svm, param_grid=svm_parameters)
svm_gs.fit(features_train, labels_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVC()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'gamma': [0.01, 0.1, ...], 'kernel': ['rbf']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"verbose verbose: int, default=0Controls the verbosity of information printed dur

In [33]:
svm_winner = svm_gs.best_estimator_
svm_winner.score(features_validation, labels_validation)

0.7191011235955056

In [34]:
svm_gs.best_params_

{'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}

### Putting all the features on the same scale: Feature scaling

SVMs are sensitive to the scale of the features (the Fare column goes up to 512, while most other columns are 0 or 1). We scale each feature to mean 0 and standard deviation 1. The scaler is fit on the training set only, and then used to transform the validation set.

In [35]:
scaler = StandardScaler()
features_train_scaled = scaler.fit_transform(features_train)
features_validation_scaled = scaler.transform(features_validation)
svm_scaled = SVC()
svm_scaled.fit(features_train_scaled, labels_train)
svm_scaled.score(features_validation_scaled, labels_validation)

0.7921348314606742

## 13.6 Using k-fold cross-validation to reuse our data as training and validation

GridSearchCV uses fivefold cross-validation by default. These are the cross-validation results.

In [36]:
pandas.DataFrame(svm_gs.cv_results_)[['param_C', 'param_gamma', 'mean_test_score', 'rank_test_score']]

,param_C,param_gamma,mean_test_score,rank_test_score
0,0.01,0.01,0.646077,17
1,0.01,0.10,0.646077,17
2,0.01,1.00,0.646077,17
3,0.01,10.00,0.646077,17
4,0.01,100.00,0.646077,17
5,0.10,0.01,0.657309,16
6,0.10,0.10,0.646077,17
7,0.10,1.00,0.646077,17
8,0.10,10.00,0.646077,17
9,0.10,100.00,0.646077,17


## Exercise 13.1

### Loading the test set

For the next two cells, please run only one of them.
- Run the first cell if you cloned the Github Repo
- Run the second cell if you opened this as a Google Colab

In [37]:
# IMPORTANT: ONLY RUN THIS CELL IF YOU HAVE CLONED THE REPO
test_data = pandas.read_csv('./test.csv')
test_data.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [ ]:
# IMPORTANT: ONLY RUN THIS CELL IF YOU ARE WORKING ON A COLAB
url = "https://raw.githubusercontent.com/luisguiserrano/manning/master/Chapter_13_End_to_end_example/test.csv"
test_data = pandas.read_csv(url)
test_data.isna().sum()

In [39]:
# Cleaning the data
test_data = test_data.drop('Cabin', axis=1)
test_data['Age'] = test_data['Age'].fillna(median_age)  # the median age from the training data (28)
# Catch! The test data has one missing fare. Let's fill it with the average fare
test_data['Fare'] = test_data['Fare'].fillna(test_data['Fare'].mean())

# One-hot encoding and binning
test_data['Age'] = pandas.cut(test_data['Age'], bins)
test_data = pandas.get_dummies(test_data, columns=['Sex', 'Embarked', 'Pclass', 'Age'], dtype=int)
test_data = test_data.drop(['Name', 'Ticket', 'PassengerId'], axis=1)

# Another small catch: the test data has no missing Embarked values, so it has no 'Embarked_U' column.
# Reindexing adds it, filled with zeros, and puts the columns in the same order as the training data.
test_data = test_data.reindex(columns=features.columns, fill_value=0)
test_data

,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_U,Pclass_1,Pclass_2,Pclass_3,"Age_(0, 10]","Age_(10, 20]","Age_(20, 30]","Age_(30, 40]","Age_(40, 50]","Age_(50, 60]","Age_(60, 70]","Age_(70, 80]"
0,0,0,7.8292,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0
1,1,0,7.0000,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0
2,0,0,9.6875,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0
3,0,0,8.6625,0,1,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0
4,1,1,12.2875,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,0,0,8.0500,0,1,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0
414,0,0,108.9000,1,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0
415,0,0,7.2500,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0
416,0,0,8.0500,0,1,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0


Now, let's check how many survivors each model predicts.

In [40]:
print('Survivors predicted in the test set (out of', len(test_data), 'passengers):')
for name, model in models.items():
    print(f'{name}: {sum(model.predict(test_data))}')

Survivors predicted in the test set (out of 418 passengers):
Logistic regression: 143
Decision tree: 142
Naive Bayes: 182
SVM: 61
Random forest: 145
Gradient boosting: 150
AdaBoost: 156


The model with the best validation accuracy, gradient boosting, predicts that 150 of the 418 passengers in the test set survived. The random forest and the decision tree, the next best models, predict 145 and 142. For comparison, 38% of the passengers in our dataset survived (342 out of 891), which would be about 160 out of 418. So a reasonable estimate is around 150 survivors.